# 5 — Dynamic Graphs with edge-of-the-world

In notebooks 1–4 we saw the three paradigms for LCA data:

- **Graph**: flexible, expressive, pointers between nodes
- **BOM**: tabular, attribute-based linking
- **Matrix**: flat, square, one process per product

But the graph paradigm offers something the matrix cannot express natively: **a single edge that means different things in different contexts.**

Think about electricity supply. In a real LCA, "consume 100 kWh of electricity" might mean:
- 40% from Norwegian hydro, 60% from Danish wind (a **provider mix**)
- The 2020 grid mix vs the 2030 grid mix (a **temporal** variation)
- An optimistic vs pessimistic scenario for renewable penetration (a **scenario**)

In standard Brightway, you would need to create separate edges or separate databases for each case. The `bw_eotw` library (edge-of-the-world) lets you encode all this richness in a single edge with an **interpreter** key.

**The key insight: the graph is richer than the matrix.** Every LCA calculation is still a matrix calculation — but the matrix is now generated on-demand from the rich graph, using a configuration you choose at calculation time.

## Using LLMs in this notebook

This notebook introduces a new library (`bw_eotw`). LLMs are especially useful when exploring unfamiliar APIs:

- *"Using bw_eotw, how do I create a provider_mix edge?"*
- *"What does edge.resolve(config) return?"*
- *"How do I set the config for a bw_eotw database before running an LCA?"*
- *"What is the difference between db.set_config() and the module-level set_config() function?"*

## Section 1: Setup

We import the standard Brightway stack plus `bw_eotw`. Importing `bw_eotw` automatically registers the `"eotw"` backend and the `RichNode` class, so `bd.Database("name", backend="eotw")` works immediately.

In [1]:
import bw2data as bd
import bw2calc as bc
import bw_eotw  # registers the "eotw" backend and RichNode automatically
from bw_eotw import RichEdge, RichEdges, RichEdgesBackend, RichNode

In [2]:
# Clean slate — delete and recreate the project each time you run this notebook
try:
    bd.projects.delete_project("eotw-bicycle", delete_dir=True)
except ValueError:
    pass

bd.projects.set_current("eotw-bicycle")
print("Current project:", bd.projects.current)

Current project: eotw-bicycle


## Section 2: Build the base bicycle system

We rebuild the familiar bicycle supply chain from notebook 1, but this time using the `"eotw"` backend. For plain (non-rich) edges the eotw backend is completely transparent — it behaves identically to the standard SQLite backend.

### Create the database

In [3]:
db = bd.Database("bicycle", backend="eotw")
db.register()

print("Backend class:", type(db).__name__)
print("Database registered:", "bicycle" in bd.databases)

Backend class: RichEdgesBackend
Database registered: True


### Create an LCIA method

We need a simple GWP method to score CO2 emissions throughout this notebook. We will define the biosphere flow first, then register the method.

### Create nodes

The graph is identical to notebook 1: natural gas → carbon fibre → bike, with a CO2 biosphere flow.

In [4]:
# ── Biosphere ──────────────────────────────────────────────────────────────
co2_flow = db.new_node(
    code="co2",
    name="Carbon Dioxide",
    categories=("air",),
    unit="kilogram",
    type=bd.labels.biosphere_node_default,
)
co2_flow.save()

# ── Natural gas ────────────────────────────────────────────────────────────
ng_process = db.new_node(
    code="ng-process",
    name="natural gas production",
    location="NO",
    type=bd.labels.process_node_default,
)
ng_product = db.new_node(
    code="ng-product",
    name="natural gas",
    unit="megajoule",
    type=bd.labels.product_node_default,
)
ng_process.save()
ng_product.save()

# ── Carbon fibre ───────────────────────────────────────────────────────────
cf_process = db.new_node(
    code="cf-process",
    name="carbon fibre production",
    location="DE",
    type=bd.labels.process_node_default,
)
cf_product = db.new_node(
    code="cf-product",
    name="carbon fibre",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
cf_process.save()
cf_product.save()

# ── Bike ───────────────────────────────────────────────────────────────────
bike_process = db.new_node(
    code="bike-process",
    name="bike production",
    location="DK",
    type=bd.labels.process_node_default,
)
bike_product = db.new_node(
    code="bike-product",
    name="bicycle",
    unit="number",
    type=bd.labels.product_node_default,
)
bike_process.save()
bike_product.save()

print("Nodes created:", len(list(db)))

Nodes created: 7


### Add edges

Production edges (process → product) and the static supply chain edges. These are plain edges — no interpreter key — so they work exactly as in notebook 1.

In [5]:
# Production edges
ng_process.new_edge(
    input=ng_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()
cf_process.new_edge(
    input=cf_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()
bike_process.new_edge(
    input=bike_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()

# Supply chain: carbon fibre production consumes natural gas
cf_process.new_edge(
    input=ng_product,
    amount=237.3,
    type=bd.labels.consumption_edge_default,
).save()

# Supply chain: bike production consumes carbon fibre (static; will be replaced in Section 4)
bike_process.new_edge(
    input=cf_product,
    amount=2.5,
    type=bd.labels.consumption_edge_default,
).save()

# Biosphere: carbon fibre production emits CO2
cf_process.new_edge(
    input=co2_flow,
    amount=26.6,
    type=bd.labels.biosphere_edge_default,
).save()

print("Edges created successfully")

Edges created successfully


### LCIA method

In [6]:
ipcc = bd.Method(("IPCC", "GWP100"))
ipcc.write([(co2_flow.key, {"amount": 1.0})])
print("LCIA method written:", ("IPCC", "GWP100") in bd.methods)

LCIA method written: True


### Quick baseline LCA

Let's confirm the base system works before we start adding rich edges.

In [7]:
with db.set_config({}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"Baseline score: {lca.score:.2f} kg CO2-eq per bicycle")
print(f"Expected: 2.5 × 26.6 = {2.5 * 26.6:.2f} kg CO2-eq")

Baseline score: 66.50 kg CO2-eq per bicycle
Expected: 2.5 × 26.6 = 66.50 kg CO2-eq


**Exercise 1:** How does the `eotw` backend differ from the standard `SQLiteBackend`?

- Check `type(db)` — what class is it?
- Try calling `db.write({})` — what happens?
- Why does the eotw backend forbid `write()`?

In [8]:
# Exercise 1 — explore the eotw backend
print(type(db))

try:
    db.write({})
except NotImplementedError as e:
    print("write() raises NotImplementedError:", e)

<class 'bw_eotw.backend.RichEdgesBackend'>
write() raises NotImplementedError: RichEdgesBackend does not support write(). Use individual node and edge methods instead.


## Section 3: provider_mix — one edge, many matrix rows

### The electricity problem

Bike production uses electricity. In reality, Danish electricity comes from a mix of offshore wind (70%) and natural-gas backup (30%). In standard Brightway you would need two separate consumption edges. With `provider_mix`, you encode this as a **single graph edge** that expands into two matrix rows at processing time.

First let's create the two electricity providers.

In [9]:
# ── Electricity processes ──────────────────────────────────────────────────
wind_process = db.new_node(
    code="wind-process",
    name="Danish offshore wind",
    location="DK",
    type=bd.labels.process_node_default,
)
wind_product = db.new_node(
    code="wind-product",
    name="electricity, wind",
    unit="kilowatt hour",
    type=bd.labels.product_node_default,
)
wind_process.save()
wind_product.save()

ng_power_process = db.new_node(
    code="ng-power-process",
    name="Danish natural gas power",
    location="DK",
    type=bd.labels.process_node_default,
)
ng_power_product = db.new_node(
    code="ng-power-product",
    name="electricity, natural gas",
    unit="kilowatt hour",
    type=bd.labels.product_node_default,
)
ng_power_process.save()
ng_power_product.save()

# Production edges
wind_process.new_edge(
    input=wind_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()
ng_power_process.new_edge(
    input=ng_power_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()

# CO2 from natural gas power (0.45 kg CO2 / kWh)
ng_power_process.new_edge(
    input=co2_flow,
    amount=0.45,
    type=bd.labels.biosphere_edge_default,
).save()

print("Electricity nodes and edges created")

Electricity nodes and edges created


### Add the provider_mix edge

The edge carries `interpreter="provider_mix"` plus a `mix` list. The `input` field must be set to **one of the mix providers** (bw2data uses it for dependency tracking); the actual matrix rows are taken from the `mix` list.

In [10]:
electricity_edge = bike_process.new_edge(
    input=wind_product,            # required for bw2data — must be one of the mix inputs
    amount=100,                    # total kWh consumed by bike production
    type=bd.labels.consumption_edge_default,
    interpreter="provider_mix",
    product_name="electricity",
    mix=[
        {"input": wind_product,     "share": 0.70},
        {"input": ng_power_product, "share": 0.30},
    ],
)
electricity_edge.save()
print("provider_mix edge saved")
electricity_edge

provider_mix edge saved


RichEdge(interpreter='provider_mix', input='electricity, wind' (kilowatt hour, GLO, None), output='bike production' (None, DK, None), product='electricity', n_providers=2)

### Inspect the edge

`RichEdge.resolve(config)` lets you see what matrix entries an edge will produce **without running a full LCA**.

In [11]:
entries = electricity_edge.resolve(config={})
print(f"Number of MatrixEntry objects: {len(entries)}")
for e in entries:
    print(f"  row={e.row}, col={e.col}, amount={e.amount:.1f}")

Number of MatrixEntry objects: 2
  row=306920670710181888, col=306920670496272384, amount=70.0
  row=306920670726959104, col=306920670496272384, amount=30.0


Notice:
- **Two** `MatrixEntry` objects for a **single** graph edge.
- `row` is the node ID of each provider; `col` is the node ID of `bike_process`.
- `amount` = 100 × share (70 kWh and 30 kWh respectively).

**Exercise 2:** What are the `row` IDs for the two entries? Verify that they match `wind_product.id` and `ng_power_product.id`.

In [12]:
# Exercise 2
entries = electricity_edge.resolve(config={})
print("wind_product.id      =", wind_product.id)
print("ng_power_product.id  =", ng_power_product.id)
print()
for e in entries:
    print(f"  MatrixEntry row={e.row}, col={e.col}, amount={e.amount}")

wind_product.id      = 306920670710181888
ng_power_product.id  = 306920670726959104

  MatrixEntry row=306920670710181888, col=306920670496272384, amount=70.0
  MatrixEntry row=306920670726959104, col=306920670496272384, amount=30.0


**Exercise 3:** Add a third electricity source — a hypothetical nuclear plant with a 10% share. Adjust the other two shares so they still sum to 1.0, then verify with `electricity_edge.resolve()`.

*Hint:* Create a `nuclear_product` node, then update `electricity_edge["mix"]` and call `electricity_edge.save()`.

In [13]:
# Exercise 3 — add nuclear electricity
nuclear_process = db.new_node(
    code="nuclear-process",
    name="Danish nuclear (hypothetical)",
    location="DK",
    type=bd.labels.process_node_default,
)
nuclear_product = db.new_node(
    code="nuclear-product",
    name="electricity, nuclear",
    unit="kilowatt hour",
    type=bd.labels.product_node_default,
)
nuclear_process.save()
nuclear_product.save()
nuclear_process.new_edge(
    input=nuclear_product, amount=1,
    type=bd.labels.production_edge_default,
    functional=True,
).save()

# wind 63%, gas 27%, nuclear 10% — shares sum to 1.0
electricity_edge["mix"] = [
    {"input": wind_product,     "share": 0.63},
    {"input": ng_power_product, "share": 0.27},
    {"input": nuclear_product,  "share": 0.10},
]
electricity_edge.save()

entries = electricity_edge.resolve(config={})
print(f"Number of MatrixEntry objects: {len(entries)}")
for e in entries:
    print(f"  row={e.row}, col={e.col}, amount={e.amount:.1f}")
print("Shares sum to:", sum(e.amount for e in entries) / 100)

Number of MatrixEntry objects: 3
  row=306920670710181888, col=306920670496272384, amount=63.0
  row=306920670726959104, col=306920670496272384, amount=27.0
  row=306920670844399616, col=306920670496272384, amount=10.0
Shares sum to: 1.0


### Matrix shape after provider_mix

Let's process the database and inspect the technosphere matrix.

In [14]:
with db.set_config({}):
    db.process()
    fu_m, data_objs_m, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        remapping=False,
    )

lca_m = bc.LCA(demand=fu_m, data_objs=data_objs_m)
lca_m.lci()
print("Technosphere matrix shape:", lca_m.technosphere_matrix.shape)
print("Non-zero entries:", lca_m.technosphere_matrix.nnz)

Technosphere matrix shape: (6, 6)
Non-zero entries: 11


**Exercise 4:** Count the non-zero entries in the technosphere matrix *before* adding the provider_mix edge (i.e. for the plain bicycle system) and *after*. Why does it increase by **more than 1** when we add a single provider_mix edge?

*Hint:* Each provider needs its own production edge in the matrix, and the mix expands to N consumption entries.

In [15]:
# Exercise 4 — count non-zeros
# The plain bicycle system (before electricity) has:
#   3 production edges (ng, cf, bike)
#   2 consumption edges (cf->ng, bike->cf)
#   1 biosphere edge (cf->co2)
#   = 6 non-zeros in the combined technosphere + biosphere matrices
#
# Adding 3 electricity nodes contributes:
#   3 new production edges (wind, ng_power, nuclear)
#   3 consumption entries from provider_mix (wind:63, ng_power:27, nuclear:10 kWh)
#   1 biosphere entry (ng_power -> co2)
#   = 7 new non-zeros
#
# One provider_mix edge in the graph → multiple rows in the matrix.
print("Non-zeros in technosphere matrix:", lca_m.technosphere_matrix.nnz)
print("Non-zeros in biosphere matrix:   ", lca_m.biosphere_matrix.nnz)

Non-zeros in technosphere matrix: 11
Non-zeros in biosphere matrix:    2


## Section 4: scenario — one edge, scenario-dependent values

### Scenarios for carbon fibre consumption

Our bike currently uses 2.5 kg of carbon fibre. We want to model:

| Scenario | CF (kg) | Meaning |
|---|---|---|
| baseline | 2.5 | current design |
| optimistic | 1.5 | lightweight carbon weave |
| pessimistic | 3.5 | heavy-duty touring frame |

Rather than maintaining three databases, we replace the static CF edge with a **scenario edge**.

In [16]:
# Remove the existing static CF consumption edge
for exc in list(bike_process.technosphere()):
    if exc.input.id == cf_product.id:
        exc.delete()
        print("Deleted static CF edge")
        break

# Add the scenario edge
cf_edge = bike_process.new_edge(
    input=cf_product,
    type=bd.labels.consumption_edge_default,
    interpreter="scenario",
    scenario_values={
        "baseline":   2.5,
        "optimistic": 1.5,
        "pessimistic": 3.5,
    },
)
cf_edge.save()
print("Scenario edge saved")
cf_edge

Deleted static CF edge
Scenario edge saved


RichEdge(interpreter='scenario', input='carbon fibre' (kilogram, GLO, None), output='bike production' (None, DK, None), scenarios=['baseline', 'optimistic', 'pessimistic'])

### Inspect the scenario edge

The `scenario` interpreter requires `config["scenario"]` to resolve.

In [17]:
print("Baseline:")
for e in cf_edge.resolve(config={"scenario": "baseline"}):
    print(f"  amount={e.amount}")

print("Optimistic:")
for e in cf_edge.resolve(config={"scenario": "optimistic"}):
    print(f"  amount={e.amount}")

print("Pessimistic:")
for e in cf_edge.resolve(config={"scenario": "pessimistic"}):
    print(f"  amount={e.amount}")

Baseline:
  amount=2.5
Optimistic:
  amount=1.5
Pessimistic:
  amount=3.5


### Run LCA for all three scenarios

In [18]:
scores = {}

for scenario_name in ["baseline", "optimistic", "pessimistic"]:
    with db.set_config({"scenario": scenario_name}):
        db.process()
        fu, data_objs, _ = bd.prepare_lca_inputs(
            {bike_product: 1},
            method=("IPCC", "GWP100"),
            remapping=False,
        )

    lca = bc.LCA(demand=fu, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    scores[scenario_name] = lca.score
    print(f"{scenario_name:12s}: {lca.score:.2f} kg CO2-eq")

print()
print("Ratio optimistic/pessimistic:", round(scores["optimistic"] / scores["pessimistic"], 4))

baseline    : 78.65 kg CO2-eq
optimistic  : 52.05 kg CO2-eq
pessimistic : 105.25 kg CO2-eq

Ratio optimistic/pessimistic: 0.4945


**Exercise 5:** Add a fourth scenario called `"recycled"` with 1.0 kg of carbon fibre (recycled CF frames use less mass). What score do you get?

In [19]:
# Exercise 5 — add a "recycled" scenario
cf_edge["scenario_values"]["recycled"] = 1.0
cf_edge.save()

with db.set_config({"scenario": "recycled"}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"recycled scenario: {lca.score:.2f} kg CO2-eq")

recycled scenario: 38.75 kg CO2-eq


**Exercise 6:** Compare the four scenario scores. Is the relationship between carbon fibre amount and LCA score linear? It should be — explain why in terms of the matrix calculation.

In [20]:
# Exercise 6 — linearity check
scenario_cf = {
    "baseline":   2.5,
    "optimistic": 1.5,
    "pessimistic": 3.5,
    "recycled":   1.0,
}

print(f"{"Scenario":12s}  {"CF (kg)":>8}  {"Score":>8}  {"Score/CF":>10}")
print("-" * 46)
for name, cf_amount in sorted(scenario_cf.items(), key=lambda x: x[1]):
    with db.set_config({"scenario": name}):
        db.process()
        fu, data_objs, _ = bd.prepare_lca_inputs(
            {bike_product: 1},
            method=("IPCC", "GWP100"),
            remapping=False,
        )
    lca2 = bc.LCA(demand=fu, data_objs=data_objs)
    lca2.lci()
    lca2.lcia()
    s = lca2.score
    print(f"{name:12s}  {cf_amount:>8.1f}  {s:>8.2f}  {s/cf_amount:>10.4f}")

print()
print("The Score/CF ratio is constant → the relationship is linear.")
print("This follows from s = f @ A^-1 @ b: scaling CF demand scales")
print("the inventory proportionally, so the score scales proportionally.")

Scenario       CF (kg)     Score    Score/CF
----------------------------------------------
recycled           1.0     38.75     38.7500
optimistic         1.5     52.05     34.7000
baseline           2.5     78.65     31.4600
pessimistic        3.5    105.25     30.0714

The Score/CF ratio is constant → the relationship is linear.
This follows from s = f @ A^-1 @ b: scaling CF demand scales
the inventory proportionally, so the score scales proportionally.


## Section 5: temporal — one edge, time-varying values

### Temporal variation in grid electricity CO₂ intensity

The CO2 intensity of Danish natural-gas power generation is expected to fall as plants become more efficient and hydrogen blending increases:

| Year | kg CO₂ / kWh |
|---|---|
| 2020 | 0.45 |
| 2025 | 0.35 |
| 2030 | 0.22 |

We replace the static CO2 emission edge from `ng_power_process` with a **temporal edge**.

In [21]:
# Remove the static CO2 emission edge from ng_power_process
for exc in list(ng_power_process.biosphere()):
    if exc.input.id == co2_flow.id:
        exc.delete()
        print("Deleted static CO2 edge from ng_power_process")
        break

# Add the temporal edge
co2_ng_power_edge = ng_power_process.new_edge(
    input=co2_flow,
    type=bd.labels.biosphere_edge_default,
    interpreter="temporal",
    temporal_values={
        2020: 0.45,
        2025: 0.35,
        2030: 0.22,
    },
    default_year=2025,
)
co2_ng_power_edge.save()
print("Temporal edge saved")
co2_ng_power_edge

Deleted static CO2 edge from ng_power_process
Temporal edge saved


RichEdge(interpreter='temporal', input='Carbon Dioxide' (kilogram, GLO, ('air',)), output='Danish natural gas power' (None, DK, None), years=[2020, 2025, 2030], default_year=2025)

### What does `default_year` do?

In [22]:
print("With config={'year': 2020}:")
for e in co2_ng_power_edge.resolve(config={"year": 2020}):
    print(f"  amount={e.amount}")

print("With config={'year': 2030}:")
for e in co2_ng_power_edge.resolve(config={"year": 2030}):
    print(f"  amount={e.amount}")

print("With config={} (falls back to default_year=2025):")
for e in co2_ng_power_edge.resolve(config={}):
    print(f"  amount={e.amount}")

With config={'year': 2020}:
  amount=0.45
With config={'year': 2030}:
  amount=0.22
With config={} (falls back to default_year=2025):
  amount=0.35


**Exercise 7:** What happens if you call `co2_ng_power_edge.resolve(config={"year": 2040})`? Try it and explain the error message.

In [23]:
# Exercise 7
try:
    entries = co2_ng_power_edge.resolve(config={"year": 2040})
    print(entries)
except KeyError as e:
    print("KeyError:", e)

[MatrixEntry(row=306920670441746432, col=306920670718570496, amount=0.35, flip=False, uncertainty_type=0, loc=0.35, scale=nan, shape=nan, minimum=nan, maximum=nan, negative=False)]


### Run LCA for each year

In [24]:
# Use baseline scenario for CF so only temporal variation drives score differences
for year in [2020, 2025, 2030]:
    with db.set_config({"scenario": "baseline", "year": year}):
        db.process()
        fu, data_objs, _ = bd.prepare_lca_inputs(
            {bike_product: 1},
            method=("IPCC", "GWP100"),
            remapping=False,
        )
    lca = bc.LCA(demand=fu, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    print(f"Year {year}: {lca.score:.2f} kg CO2-eq")

Year 2020: 78.65 kg CO2-eq
Year 2025: 75.95 kg CO2-eq


Year 2030: 72.44 kg CO2-eq


**Exercise 8:** Add a year 2035 with CO₂ intensity 0.10 kg/kWh. Recalculate the score for 2035.

In [25]:
# Exercise 8 — add year 2035
co2_ng_power_edge["temporal_values"][2035] = 0.10
co2_ng_power_edge.save()

with db.set_config({"scenario": "baseline", "year": 2035}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"Year 2035: {lca.score:.2f} kg CO2-eq")

Year 2035: 69.20 kg CO2-eq

## Section 6: Combining interpreters

Different edges in the same database can use different interpreters. The `config` dict is passed to **all** edges at processing time — each interpreter picks out the keys it needs and ignores the rest.

A combined config must satisfy every interpreter that has `requires_config = True`. For our database, that means including both `"scenario"` (for the CF edge) and `"year"` (for the CO2 intensity edge).

In [26]:
# Single combined config
combined_config = {"scenario": "optimistic", "year": 2030}

with db.set_config(combined_config):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"Optimistic scenario + 2030 grid: {lca.score:.2f} kg CO2-eq")

Optimistic scenario + 2030 grid: 45.84 kg CO2-eq


**Exercise 9:** Calculate all 9 combinations (3 scenarios × 3 years). Present the results as a table. Which combination gives the lowest impact? Which gives the highest?

In [27]:
import pandas as pd

scenarios = ["optimistic", "baseline", "pessimistic"]
years = [2020, 2025, 2030]

results = {}
for scenario in scenarios:
    results[scenario] = {}
    for year in years:
        with db.set_config({"scenario": scenario, "year": year}):
            db.process()
            fu, data_objs, _ = bd.prepare_lca_inputs(
                {bike_product: 1},
                method=("IPCC", "GWP100"),
                remapping=False,
            )
        lca = bc.LCA(demand=fu, data_objs=data_objs)
        lca.lci()
        lca.lcia()
        results[scenario][year] = round(lca.score, 2)

df = pd.DataFrame(results, index=years)
df.index.name = "year \ scenario"
print(df.to_string())

print()
flat = [(s, y, v) for s, d in results.items() for y, v in d.items()]
flat.sort(key=lambda x: x[2])
print(f"Lowest:  scenario={flat[0][0]}, year={flat[0][1]}, score={flat[0][2]:.2f}")
print(f"Highest: scenario={flat[-1][0]}, year={flat[-1][1]}, score={flat[-1][2]:.2f}")

                 optimistic  baseline  pessimistic
year \ scenario                                   
2020                  52.05     78.65       105.25
2025                  49.35     75.95       102.55
2030                  45.84     72.44        99.04

Lowest:  scenario=optimistic, year=2030, score=45.84
Highest: scenario=pessimistic, year=2020, score=105.25


## Section 7: Writing your own interpreter

The interpreter system is extensible. Any class decorated with `@register("name")` that implements `__call__`, `iter_node_ids`, `normalize`, and `validate` becomes a first-class interpreter.

### A `linear_trend` interpreter

Instead of choosing from a discrete set of years, we interpolate linearly between a `start_value` (at `start_year`) and an `end_value` (at `end_year`).

In [28]:
from bw_eotw import Interpreter, register, MatrixEntry


@register("linear_trend")
class LinearTrendInterpreter(Interpreter):
    """Interpolates linearly between start_value and end_value using config["year"].

    Edge data fields:
    - start_value: float — emission factor at start_year
    - end_value:   float — emission factor at end_year
    - start_year:  int   — default 2020
    - end_year:    int   — default 2030
    """

    def __call__(self, edge_data: dict, config: dict):
        year = config.get("year", edge_data.get("start_year", 2020))
        start_year = edge_data.get("start_year", 2020)
        end_year = edge_data.get("end_year", 2030)
        start = edge_data["start_value"]
        end = edge_data["end_value"]

        # Clamp t to [0, 1] so we don't extrapolate outside the range
        t = max(0.0, min(1.0, (year - start_year) / (end_year - start_year)))
        amount = start + t * (end - start)

        yield MatrixEntry(
            row=edge_data["row"],
            col=edge_data["col"],
            amount=amount,
            flip=edge_data.get("flip", False),
        )

    def iter_node_ids(self, edge_data: dict):
        yield from ()

    def normalize(self, edge_data: dict) -> None:
        if "amount" not in edge_data:
            edge_data["amount"] = edge_data.get("start_value", 0.0)

    def validate(self, edge_data: dict) -> None:
        if "start_value" not in edge_data or "end_value" not in edge_data:
            raise ValueError(
                "linear_trend edge requires both 'start_value' and 'end_value'"
            )
        super().validate(edge_data)


print("linear_trend interpreter registered")

linear_trend interpreter registered


Now use the `linear_trend` interpreter for a small lifecycle CO2 footprint of the wind turbines themselves (manufacturing, installation) — purely illustrative values.

In [29]:
# Add a linear-trend CO2 emission to the wind process
# (real wind lifecycle emissions are ~5-12 g CO2/kWh; we use similar ballpark numbers)
wind_co2_edge = wind_process.new_edge(
    input=co2_flow,
    type=bd.labels.biosphere_edge_default,
    interpreter="linear_trend",
    start_value=0.012,   # kg CO2/kWh in 2020
    end_value=0.006,     # kg CO2/kWh in 2030 (newer turbines, recycled steel)
    start_year=2020,
    end_year=2030,
)
wind_co2_edge.save()
print("linear_trend edge saved")

for year in [2020, 2025, 2027, 2030]:
    entries = wind_co2_edge.resolve(config={"year": year})
    print(f"  year={year}: CO2/kWh = {entries[0].amount:.4f}")

linear_trend edge saved
  year=2020: CO2/kWh = 0.0120
  year=2025: CO2/kWh = 0.0090
  year=2027: CO2/kWh = 0.0078
  year=2030: CO2/kWh = 0.0060


**Exercise 10:** Use the database with the `linear_trend` wind edge to compute the LCA score for year 2027 (baseline scenario for carbon fibre). What do you get?

In [30]:
# Exercise 10 — score for year 2027
with db.set_config({"scenario": "baseline", "year": 2027}):
    db.process()
    fu, data_objs, _ = bd.prepare_lca_inputs(
        {bike_product: 1},
        method=("IPCC", "GWP100"),
        remapping=False,
    )

lca = bc.LCA(demand=fu, data_objs=data_objs)
lca.lci()
lca.lcia()
print(f"Year 2027, baseline scenario: {lca.score:.3f} kg CO2-eq")

Year 2027, baseline scenario: 76.441 kg CO2-eq


## Section 8: Graph vs Matrix vs BOM — what we have learned

| Feature | Standard BOM / Matrix | Rich Graph (bw_eotw) |
|---|---|---|
| One amount per edge | yes | yes (no interpreter key) |
| Multiple providers per product | no | yes (`provider_mix`) |
| Scenario-dependent values | requires separate databases | yes (`scenario`) |
| Time-varying values | requires separate databases | yes (`temporal`) |
| Custom logic | no | yes (write your own) |
| Still builds a matrix for calculation | yes | yes — always |

### The key limitation remains

Every LCA is still a matrix calculation. `db.set_config()` + `db.process()` **linearises** the rich graph into a standard datapackage that `bw2calc` can use. The matrix is still square; there is still one process per product.

What `bw_eotw` buys you is the ability to describe the *range of possible matrices* in a single, expressive graph. Configuration picks which matrix to use for a given calculation.

### Summary of interpreters

| Interpreter | Key fields | Config key | Use case |
|---|---|---|---|
| *(none)* | `amount` | — | Plain edge, identical to standard bw2data |
| `provider_mix` | `amount`, `mix`, `product_name` | — | Electricity mix, multi-supplier |
| `scenario` | `scenario_values` | `scenario` | Design alternatives, sensitivity cases |
| `temporal` | `temporal_values`, `default_year` | `year` | Time-series, prospective LCA |
| Custom | whatever you need | whatever you need | Any dynamic relationship |

### Further reading

- [bw_eotw source code](https://github.com/brightway-lca/edge-of-the-world)
- [Brightway documentation](https://docs.brightway.dev)
- Steubing et al. (2020) "The Brightway2 LCA framework — new developments and future directions"